# Training with both RNA-Seq and Microarray data
We have seen that there is significant performance drop when training with both. Here we will go through all the filtering steps and evaluate how much data we have with different filtering thresholds. 

In [31]:
import scanpy as sc
import sys
sys.path.append("../../")
from src.training import helpers as tr_h
adata = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-09-12-01/data.h5ad")


In [2]:
dict(adata.obs.groupby("dataset", observed=True)['celltype'].count()).items()

dict_items([('E-MEXP-3097', 5), ('E-MTAB-567', 27), ('E-MTAB-1030', 18), ('E-MTAB-1132', 489), ('E-MTAB-1516', 36), ('E-MTAB-1791', 192), ('E-MTAB-2312', 12), ('E-MTAB-2976', 8), ('E-MTAB-3338', 38), ('E-MTAB-3658', 8), ('E-MTAB-3664', 64), ('E-MTAB-3950', 98), ('E-MTAB-4054', 98), ('E-MTAB-4257', 34), ('E-MTAB-4586', 64), ('E-MTAB-4902', 8), ('E-MTAB-5385', 73), ('E-MTAB-5464', 66), ('E-MTAB-5616', 22), ('E-MTAB-5638', 39), ('E-MTAB-5678', 26), ('E-MTAB-6413', 21), ('E-MTAB-6490', 30), ('E-MTAB-7140', 7), ('E-MTAB-7145', 4), ('E-MTAB-7340', 15), ('E-MTAB-7406', 15), ('E-MTAB-7671', 7), ('E-MTAB-7860', 192), ('E-MTAB-7876', 57), ('E-MTAB-8448', 412), ('E-MTAB-8455', 45), ('E-MTAB-8887', 30), ('E-MTAB-9045', 20), ('E-MTAB-10190', 160), ('E-MTAB-10318', 70), ('E-MTAB-10604', 40), ('E-MTAB-10703', 134), ('E-MTAB-10916', 26), ('E-MTAB-11029', 9), ('E-MTAB-11240', 102), ('E-MTAB-11326', 107), ('E-MTAB-11415', 217), ('E-MTAB-11910', 40), ('E-MTAB-11919', 20), ('E-MTAB-12014', 20), ('E-MTAB-1

In [ ]:
# drop duplicates
adata.obs["celltype"] = adata.obs["doid_id"]
adata.obs["sample_id"] = [x.split(".")[1] for x in adata.obs["ids"]]
print(adata.shape)
adata_drop = adata[~adata.obs.duplicated(subset=['sample_id','celltype'])]
print(adata_drop.shape)

# drop duplicate samples
adata.obs[~adata.obs.duplicated('sample_id', keep='first')]

(121142, 20608)
(100543, 20608)


NameError: name 'X_log1p' is not defined

In [115]:
adata_drop.shape

(100451, 20608)

In [116]:
adata_drop.obs.duplicated(subset=['sample_id']).sum()

1611

In [114]:
adata_drop.obs.groupby("sample_id")["celltype"]

Removed samples in multiple datasets, new shape: (100451, 20608)
Removed duplicate samples, new shape: (100451, 20608)


In [ ]:

adata_drop.obs[adata_drop.obs["dataset"] == "GSE20257"]["sample_id"].to_list()

['GSM101107', 'GSM364037']

In [73]:
adata_drop.obs[adata_drop.obs["dataset"] == "GSE20257"]

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,sample_id
65797,DSA05794.GSM101107.Control,GSE20257,GSE20257,1356,1356,DSA05794,Epithelium,17046,Control,Control,Chronic Obstructive Pulmonary Disease,Microarray,D,Control,Control,Control,GSM101107
65858,DSA05794.GSM364037.Case,GSE20257,GSE20257,1356,1356,DSA05794,Epithelium,17046,Chronic Obstructive Pulmonary Disease,DOID:3083,Chronic Obstructive Pulmonary Disease,Microarray,D,DOID:3083,DOID:3083,chronic obstructive pulmonary disease,GSM364037


In [37]:
adata_drop.obs.groupby("dataset", observed=True)['celltype'].count()

dataset
E-MEXP-3097       5
E-MTAB-567       27
E-MTAB-1030      18
E-MTAB-1132     243
E-MTAB-1516      36
               ... 
TCGA-LUSC       550
TCGA-PAAD       182
TCGA-PRAD       551
TCGA-STAD       407
TCGA-THCA      1067
Name: celltype, Length: 2136, dtype: int64

In [ ]:
adata_drop_dis = adata_drop[adata_drop.obs['disease'] != 'Control']



In [98]:
pair_counts[pair_counts<2]

celltype   dataset 
DOID:3083  GSE20257    1
Name: n, dtype: int64

In [101]:
pair_counts = (
        adata.obs.groupby(["celltype", "dataset"], observed=True)
                .size()
                .rename("n")
    )
_datasets_control_passed = pair_counts[pair_counts >= 2].index

['E-MEXP-3097',
 'E-MTAB-567',
 'E-MTAB-1030',
 'E-MTAB-1132',
 'E-MTAB-1516',
 'E-MTAB-1791',
 'E-MTAB-2312',
 'E-MTAB-2976',
 'E-MTAB-3338',
 'E-MTAB-3658',
 'E-MTAB-3664',
 'E-MTAB-3950',
 'E-MTAB-4054',
 'E-MTAB-4257',
 'E-MTAB-4586',
 'E-MTAB-4902',
 'E-MTAB-5385',
 'E-MTAB-5464',
 'E-MTAB-5616',
 'E-MTAB-5638',
 'E-MTAB-5678',
 'E-MTAB-6413',
 'E-MTAB-6490',
 'E-MTAB-7140',
 'E-MTAB-7145',
 'E-MTAB-7340',
 'E-MTAB-7406',
 'E-MTAB-7671',
 'E-MTAB-7860',
 'E-MTAB-7876',
 'E-MTAB-8448',
 'E-MTAB-8455',
 'E-MTAB-8887',
 'E-MTAB-9045',
 'E-MTAB-10190',
 'E-MTAB-10318',
 'E-MTAB-10604',
 'E-MTAB-10703',
 'E-MTAB-10916',
 'E-MTAB-11029',
 'E-MTAB-11240',
 'E-MTAB-11326',
 'E-MTAB-11415',
 'E-MTAB-11910',
 'E-MTAB-11919',
 'E-MTAB-12014',
 'E-MTAB-12184',
 'E-MTAB-12252',
 'E-MTAB-12309',
 'E-MTAB-12631',
 'GSE420',
 'GSE474',
 'GSE475',
 'GSE593',
 'GSE833',
 'GSE860',
 'GSE1010',
 'GSE1297',
 'GSE1299',
 'GSE1377',
 'GSE1420',
 'GSE1551',
 'GSE1650',
 'GSE1710',
 'GSE1719',
 'GSE1724',

In [58]:
_dataset_count = adata.obs.groupby("dataset", observed=True)['celltype'].nunique()

_dataset_count[_dataset_count >= 6]

dataset
GSE3307       7
GSE42656      6
GSE88805      8
GSE89843      6
GSE100150     8
GSE123086    10
GSE174302     6
Name: celltype, dtype: int64

In [59]:
valid_pairs

MultiIndex([(     'Control', 'E-MEXP-3097'),
            (     'Control',  'E-MTAB-567'),
            (     'Control', 'E-MTAB-1030'),
            (     'Control', 'E-MTAB-1132'),
            (     'Control', 'E-MTAB-1516'),
            (     'Control', 'E-MTAB-1791'),
            (     'Control', 'E-MTAB-2312'),
            (     'Control', 'E-MTAB-2976'),
            (     'Control', 'E-MTAB-3338'),
            (     'Control', 'E-MTAB-3658'),
            ...
            ('DOID:0081087',   'GSE142698'),
            ('DOID:0081087',   'GSE171654'),
            ('DOID:0081087',   'GSE175701'),
            ('DOID:0081087',   'GSE183817'),
            ('DOID:0081087',   'GSE197907'),
            ('DOID:0081087',   'GSE198781'),
            ('DOID:0111394',    'GSE23075'),
            ('DOID:0111394',    'GSE32154'),
            ('DOID:0111535',    'GSE48120'),
            ('DOID:0111535',    'GSE48129')],
           names=['celltype', 'dataset'], length=4497)

In [104]:
import importlib
importlib.reload(tr_h)

<module 'src.training.helpers' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/exp/../../src/training/helpers.py'>

In [106]:
adata_f_1 = tr_h.clean_adata_qc(adata_drop,n_samples=2, n_dt=2)

Nº of datasets with +2 control samples: 2364
Nº of datasets with +2 disease samples: 2130
Nº of datasets with +2 samples (control and disease): 2128
adata shape after filtering datasets with +2 samples: (100404, 20608)
Nº of passed diseases +2 294/ 296
Nº of passed dsaids +2 3815/ 3905


In [25]:
adata_f_1.shape

(121142, 20608)

In [29]:
adata_f_2 = tr_h.clean_adata_qc_new(adata,n_samples=2, n_dt=2)

Nº of datasets with +2 control samples: 0
Nº of datasets with +2 disease samples: 169
Nº of datasets with +2 samples (both): 0
adata shape after dataset filter: (0, 20608)
Removed 0 cells from pairs with <2 samples.
All cells filtered by pair-level constraint.


In [9]:
valid_pairs

MultiIndex([(     'Control', 'E-MEXP-3097'),
            (     'Control',  'E-MTAB-567'),
            (     'Control', 'E-MTAB-1030'),
            (     'Control', 'E-MTAB-1132'),
            (     'Control', 'E-MTAB-1516'),
            (     'Control', 'E-MTAB-1791'),
            (     'Control', 'E-MTAB-2312'),
            (     'Control', 'E-MTAB-2976'),
            (     'Control', 'E-MTAB-3338'),
            (     'Control', 'E-MTAB-3658'),
            ...
            ('DOID:0081087',   'GSE142698'),
            ('DOID:0081087',   'GSE171654'),
            ('DOID:0081087',   'GSE175701'),
            ('DOID:0081087',   'GSE183817'),
            ('DOID:0081087',   'GSE197907'),
            ('DOID:0081087',   'GSE198781'),
            ('DOID:0111394',    'GSE23075'),
            ('DOID:0111394',    'GSE32154'),
            ('DOID:0111535',    'GSE48120'),
            ('DOID:0111535',    'GSE48129')],
           names=['doid_id', 'dataset'], length=4512)

In [12]:
import pandas as pd
mi = pd.MultiIndex.from_frame(adata.obs[[disease_label, group_label]])

mask_pairs = mi.isin(valid_pairs)

In [15]:
import numpy as np
np.sum(mask_pairs)

121142

In [16]:
len(mask_pairs)

121142

In [17]:
adata.n_obs

121142

In [7]:
adata.obs.groupby("dataset")["doid_id"].count()

/tmp/ipykernel_456335/2037796854.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata.obs.groupby("dataset")["doid_id"].count()


dataset
E-MEXP-3097       5
E-MTAB-567       27
E-MTAB-1030      18
E-MTAB-1132     489
E-MTAB-1516      36
               ... 
TCGA-LUSC       550
TCGA-PAAD       182
TCGA-PRAD       551
TCGA-STAD       407
TCGA-THCA      1698
Name: doid_id, Length: 2140, dtype: int64

In [73]:
adata_2 = sc.read("/aloy/home/ddalton/projects/scGPT_playground/data/pp_data-25-08-14-01/data.h5ad")

In [74]:
# what datasets am I missing ?

d2 = set(adata_2.obs["dataset_id"].unique())
d1 = set(adata.obs["dataset_id"].unique())

In [97]:
d_missing = (d2-d1)
adata_2.obs.query("dataset_id in @d_missing")["dsaid"].value_counts()

dsaid
DSA08970    585
DSA08971    583
DSA08981    551
DSA08973    550
DSA08960    546
           ... 
DSA02758      0
DSA02757      0
DSA02755      0
DSA02731      0
DSA02780      0
Name: count, Length: 1159, dtype: int64

In [ ]:
mask_genes = tr_h.get_top_k_most_present_genes(
    adata, k=3501
)

In [106]:
import importlib
importlib.reload(pp)

<module 'src.preprocessing.pipeline' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/exp/../../src/preprocessing/pipeline.py'>

In [109]:
from src.preprocessing import pipeline as pp 

bp = pp.bulk_processing()

for ds in d_missing:
    _dtype =  bp._classify_expr(adata_2.X[adata_2.obs["dataset_id"] == ds])
    print(ds, _dtype)

TCGA-PAAD log
GSE184053 log
TCGA-HNSC log
TCGA-LUSC log
GSE127853 log
TCGA-ESCA log
TCGA-PRAD log
GSE137327 log
GSE114245 log
TCGA-STAD log
TCGA-COAD log
TCGA-LUAD log
TCGA-GBM log
TCGA-CHOL log
GSE38003 log
TCGA-LIHC log
TCGA-BLCA log


In [ ]:
b_p = pp.bulk_processing(processing="linear", do_z_transform=False, agg_genes="median")

# process ids
d_types, df = b_p.get_processed_exp_prof(d_missing)

KeyboardInterrupt: 

In [ ]:
import numpy as np
np.power(2, (adata_2.X[adata_2.obs["dataset_id"] == ds])) - 1


AttributeError: 'numpy.ndarray' object has no attribute 'columns'

In [ ]:
df

In [105]:
_X = adata_2.X[adata_2.obs["dataset_id"] == list(d_missing)[0]]
_X = np.where(np.isnan(_X), 0, _X )
_X.max()

16.728246612790176

In [96]:
adata_2.obs["dataset_id"] in d_missing

TypeError: unhashable type: 'Series'

In [7]:
adata = adata[:, mask_genes]

In [53]:
import numpy as np

nan_thr= 0.9

non_nan_mask = ~np.isnan(adata.X)  & ~(adata.X==0) 
non_nan_mask_pct = np.sum(non_nan_mask, axis=1) / adata.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples_nan = non_nan_mask_pct >= nan_thr 

print(f"{nan_thr} Keeping {np.sum(mask_samples_nan)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples_nan)/adata.X.shape[0]*100:.2f}%)")


zero_thr = 0.5
non_zero_mask = ~(adata.X==0) 
non_zero_mask_pct = np.sum(non_zero_mask, axis=1) / adata.X.shape[1]

# mask samples that have less than 30% non-NaN values
mask_samples_zero = non_zero_mask_pct >= zero_thr 

print(f"{zero_thr} Keeping {np.sum(mask_samples_zero)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples_zero)/adata.X.shape[0]*100:.2f}%)")

mask_samples_comb = mask_samples_nan & mask_samples_zero
print(f"Combined: Keeping {np.sum(mask_samples_comb)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples_comb)/adata.X.shape[0]*100:.2f}%)")

0.9 Keeping 100610 samples out of 111082 (90.57%)
0.5 Keeping 109791 samples out of 111082 (98.84%)
Combined: Keeping 100610 samples out of 111082 (90.57%)


In [9]:
for pct_thr in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    non_zero_non_nan_mask = ~(adata.X==0) 
    non_zero_non_nan_mask_pct = np.sum(non_zero_non_nan_mask, axis=1) / adata.X.shape[1]

    # mask samples that have less than 30% non-NaN values
    mask_samples = non_zero_non_nan_mask_pct >= pct_thr 

    print(f"{pct_thr} Keeping {np.sum(mask_samples)} samples out of {adata.X.shape[0]} ({np.sum(mask_samples)/adata.X.shape[0]*100:.2f}%)")

0.3 Keeping 110630 samples out of 111082 (99.59%)
0.4 Keeping 110355 samples out of 111082 (99.35%)
0.5 Keeping 109791 samples out of 111082 (98.84%)
0.6 Keeping 109061 samples out of 111082 (98.18%)
0.7 Keeping 108535 samples out of 111082 (97.71%)
0.8 Keeping 108078 samples out of 111082 (97.30%)
0.9 Keeping 106747 samples out of 111082 (96.10%)


In [ ]:

# apply the mask to the AnnData object
adata[mask_samples, :].shape()


In [18]:
filtered_adata  = tr_h.clean_adata_qc(adata,n_samples=1, n_dt=4)


Nº of datasets with +1 control samples: 2111
Nº of datasets with +1 disease samples: 2111
Nº of datasets with +1 samples (control and disease): 2111
adata shape after filtering datasets with +1 samples: (111082, 3501)
Nº of passed diseases 142/ 292
Nº of passed dsaids 3395/ 3927


In [15]:
filtered_adata.obs["doid_id"].nunique()

196

In [45]:
adata_f.obs.query('doid_id == "DOID:1578"')["dataset_id"].unique()

['GSE166059', 'GSE40839', 'GSE81292']
Categories (1958, object): ['E-MEXP-3097', 'E-MTAB-567', 'E-MTAB-1030', 'E-MTAB-1132', ..., 'GSE224056', 'GSE225904', 'GSE226869', 'GSE227329']

In [50]:
adata_f.obs.query("dataset_id == 'GSE40839'")["doid_id"].unique()

['Control', 'DOID:1578']
Categories (196, object): ['Control', 'DOID:235', 'DOID:289', 'DOID:299', ..., 'DOID:0060224', 'DOID:0060901', 'DOID:0080199', 'DOID:0081087']

In [51]:
import importlib
importlib.reload(tr_h)

<module 'src.training.helpers' from '/aloy/home/ddalton/projects/scGPT_playground/notebooks/exp/../../src/training/helpers.py'>

In [72]:
tr_h.clean_adata_qc(adata[adata.obs["library"]=="RNA-Seq"], n_samples=1, n_dt=2)


Nº of datasets with +1 control samples: 712
Nº of datasets with +1 disease samples: 712
Nº of datasets with +1 samples (control and disease): 712
adata shape after filtering datasets with +1 samples: (31606, 3501)
Nº of passed diseases 123/ 218
Nº of passed dsaids 1120/ 1261


View of AnnData object with n_obs × n_vars = 28425 × 3501
    obs: 'ids', 'dataset', 'dataset_id', 'batch', 'batch_id', 'dsaid', 'tissue', 'n_genes', 'disease', 'celltype', 'disease_study', 'library', 'doid_study', 'doid_id', 'do_id', 'doid_disease'
    var: 'gene_symbols', 'gene_name', 'index'

In [61]:

adata_q = adata[mask_samples, :]

adata_q = adata_q[adata_q.obs["library"]== "RNA-Seq"]
print(f"Nº of diseases before filtering: {adata_q.obs['doid_id'].nunique()}")

adata_f = tr_h.clean_adata_qc(adata_q, n_samples=1, n_dt=2)
print(f"Nº of diseases after filtering: {adata_f.obs['doid_id'].nunique()}")


# split
df_obs = adata_f.obs
test_obs = tr_h.split_stratified(
    df=df_obs,
    y_label="doid_id",  # should ALWAYS be on DOID! - OR  celltype be DOID! 
    group_label="dataset_id",
    split_size=10,
    seed=42,
)
tr_h.report_split(test_obs, disease_label="doid_id")





valid_obs = tr_h.split_stratified(
    df=test_obs[test_obs["test_split_1"]==0],
    y_label="doid_id",  # should ALWAYS be on DOID! - OR  celltype be DOID! 
    group_label="dataset_id",
    split_size=10,
    seed=42,
    new_label="valid_split_1"
)
tr_h.report_split(valid_obs, disease_label="doid_id", split_label="valid_split_1")



Nº of diseases before filtering: 215
Nº of datasets with +1 control samples: 661
Nº of datasets with +1 disease samples: 662
Nº of datasets with +1 samples (control and disease): 655
adata shape after filtering datasets with +1 samples: (27413, 3501)
Nº of passed diseases 118/ 214
Nº of passed dsaids 1024/ 1168
Nº of diseases after filtering: 119
df shape: (24492, 16)
df diseases: (13271, 16)
All Labels: 119 ['Control', 'DOID:0050156', 'DOID:0050700', 'DOID:0060161', 'DOID:0060488', 'DOID:0080199', 'DOID:0081087', 'DOID:10126', 'DOID:1024', 'DOID:10286', 'DOID:1040', 'DOID:10584', 'DOID:10591', 'DOID:10608', 'DOID:10652', 'DOID:10763', 'DOID:10871', 'DOID:10923', 'DOID:10941', 'DOID:11335', 'DOID:11555', 'DOID:11589', 'DOID:11612', 'DOID:11714', 'DOID:11722', 'DOID:11725', 'DOID:11727', 'DOID:11934', 'DOID:11963', 'DOID:11984', 'DOID:12365', 'DOID:12377', 'DOID:12449', 'DOID:12704', 'DOID:12858', 'DOID:12894', 'DOID:12930', 'DOID:1312', 'DOID:13241', 'DOID:13922', 'DOID:14250', 'DOID:1

In [39]:
adata_f.obs.groupby("doid_id")["dataset"].nunique().sort_values()

/tmp/ipykernel_153703/2624498825.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  adata_f.obs.groupby("doid_id")["dataset"].nunique().sort_values()


doid_id
DOID:0060901       3
DOID:676           3
DOID:0060224       3
DOID:0050427       3
DOID:1107          3
                ... 
DOID:10652        59
DOID:9074         63
DOID:2841         64
DOID:14330        66
Control         1958
Name: dataset, Length: 196, dtype: int64

In [42]:
df = adata_f.obs
y_label = "doid_id" 
group_label = "dataset_id"

# (tiny guard) each label must span ≥2 datasets to appear in both splits
_df_diseases = df[df[y_label] != "Control"]
print(f"df diseases: {_df_diseases.shape}")

ds_per_label = _df_diseases.groupby(y_label, observed=True)[group_label].nunique()
if (ds_per_label < 2).any():
    print(ds_per_label[ds_per_label < 2])
    raise ValueError("Some labels occur in <2 datasets; cannot place them in both splits.")


df diseases: (57716, 16)


In [28]:
test_obs

,ids,dataset,dataset_id,batch,batch_id,dsaid,tissue,n_genes,disease,celltype,disease_study,library,doid_study,doid_id,do_id,doid_disease,test_split_1
9,DSA00006.GSM3596906.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
10,DSA00006.GSM3596907.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
11,DSA00006.GSM3596908.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
12,DSA00006.GSM3596909.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
13,DSA00006.GSM3596910.Control,GSE126342,GSE126342,1512,1512,DSA00006,Skeletal muscle,19402,Control,Control,Myotonic Dystrophy Type 1,RNA-Seq,D,Control,Control,Control,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
111077,DSA10302.GSM139471.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0
111078,DSA10302.GSM139472.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0
111079,DSA10302.GSM139473.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0
111080,DSA10302.GSM139474.Case,GSE6008,GSE6008,1354,1354,DSA10302,Ovary,12004,Ovarian Tumor,Ovarian Tumor,Ovarian Tumor,Microarray,D,DOID:2394,DOID:2394,ovarian cancer,0


In [20]:
adata_f.obs["doid_id"].nunique()

143

In [3]:
# quality control cleaning - enough samples and datasets
print("BEFORE QC - ADATA SHAPE:", adata.shape)
adata = tr_h.clean_adata_qc(adata,n_samples=1, n_dt=3)
print("AFTER QC - ADATA SHAPE:", adata.shape)


BEFORE QC - ADATA SHAPE: (121142, 20608)
Nº of datasets with +1 control samples: 2140
Nº of datasets with +1 disease samples: 2140
Nº of datasets with +1 samples (control and disease): 2140
adata shape after filtering datasets with +1 samples: (121142, 20608)
Nº of passed diseases 196/ 296
Nº of passed dsaids 3687/ 3986
AFTER QC - ADATA SHAPE: (112328, 20608)


In [11]:
import importlib
importlib.reload(tr_h)


split = 1


df_obs = adata.obs
new_obs = tr_h.split_stratified(
    df=df_obs,
    y_label="doid_id",
    group_label="dataset_id",
    split_size=10,
    seed=42,
)
tr_h.report_split(new_obs, disease_label="doid_id")


df shape: (112328, 16)
df diseases: (64108, 16)
All Labels: 197 ['Control', 'DOID:0050156', 'DOID:0050427', 'DOID:0050458', 'DOID:0050589', 'DOID:0050700', 'DOID:0050865', 'DOID:0050902', 'DOID:0060058', 'DOID:0060161', 'DOID:0060224', 'DOID:0060901', 'DOID:0080199', 'DOID:0081087', 'DOID:10223', 'DOID:1024', 'DOID:10241', 'DOID:10286', 'DOID:1040', 'DOID:10534', 'DOID:10584', 'DOID:10591', 'DOID:10608', 'DOID:10652', 'DOID:10763', 'DOID:10871', 'DOID:10923', 'DOID:10941', 'DOID:10976', 'DOID:1107', 'DOID:11100', 'DOID:11265', 'DOID:11335', 'DOID:11555', 'DOID:11589', 'DOID:11612', 'DOID:11714', 'DOID:11722', 'DOID:11723', 'DOID:11725', 'DOID:11727', 'DOID:11729', 'DOID:11934', 'DOID:11984', 'DOID:1206', 'DOID:12132', 'DOID:12177', 'DOID:12205', 'DOID:12217', 'DOID:12236', 'DOID:12306', 'DOID:12365', 'DOID:12377', 'DOID:12449', 'DOID:12689', 'DOID:12704', 'DOID:12858', 'DOID:12894', 'DOID:12930', 'DOID:1312', 'DOID:1319', 'DOID:13223', 'DOID:13241', 'DOID:13378', 'DOID:13515', 'DOID:13

In [ ]:

# update observations with new column split
adata.obs = new_obs


# seperate data
adata_test = adata[adata.obs[f"test_split_{split}"] == 1].copy()
adata = adata[adata.obs[f"test_split_{split}"] == 0].copy()
